# ENERGIZE NILM — TFLite INT8 + Edge TPU Pipeline (TCN)

This notebook converts the **fine-tuned pruned TCN** (output of `03_pruning.ipynb`) to
**TFLite full-integer INT8** and compiles it for the **Google Coral Edge TPU**.

Pipeline:
1. **Rebuild** pruned PyTorch TCN and load fine-tuned weights
2. **Re-implement** the same architecture in TF/Keras (Functional API)
3. **Transfer weights** — map PyTorch `[out, in, kernel]` → TF `[kernel, in, out]`
4. **Validate** — compare TF vs PyTorch predictions (MAE diff should be < 1 mW)
5. **Evaluate TF/Keras float32** — confirm real-world NILM performance on test set before quantizing
6. **Convert** to TFLite full-integer INT8 with representative dataset calibration
7. **Compile** with `edgetpu_compiler` — prints op-coverage report
8. **Evaluate** quantized model on the test set via TFLite CPU interpreter
9. **Export** — append results row to existing Excel, save `.tflite` artifact

> Architecture note: `CausalConv1d` in PyTorch ≡ `Conv1D(padding='causal')` in TF.  
> The gated block (ReLU × Sigmoid) maps directly; skip concatenation is channel-wise.

---
## Google Colab Setup
1. Upload your `ENERGIZE` folder to Google Drive
2. Run the cell below first and edit `DRIVE_PROJECT_PATH`

In [2]:
# ============================================================================
# COLAB SETUP â run this cell first
# ============================================================================
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'torch_pruning', 'openpyxl', 'tensorflow'])

    # =========================================================================
    DRIVE_PROJECT_PATH = '/content/drive/MyDrive/ENERGIZE'  # <-- EDIT THIS
    # =========================================================================

    import os
    from pathlib import Path
    project_root = Path(DRIVE_PROJECT_PATH)
    if not project_root.exists():
        raise FileNotFoundError(f'Project folder not found: {project_root}')
    os.chdir(project_root)
    sys.path.insert(0, str(project_root))
    print(f'Project root: {project_root}')
else:
    import os
    from pathlib import Path
    project_root = Path(os.getcwd()).parent
    sys.path.insert(0, str(project_root))
    print(f'Running locally. Project root: {project_root}')

Running locally. Project root: c:\Users\s.athanasoulias\OneDrive - Accenture\Desktop\ENERGIZE-Edge-Optimized-NILM-Framework


## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import torch

import tensorflow as tf

from src_pytorch import (
    SimpleNILMDataLoader,
    set_seeds,
    get_model_stats,
    compute_status,
    compute_metrics,
    param_ratio_to_channel_ratio,
    get_appliance_params,
    get_model_config,
    save_pipeline_results,
)
from src_pytorch.quantizer import (
    # TCN
    rebuild_pruned_tcn,
    read_pruned_channels,
    build_tcn_keras,
    transfer_weights,
    # CNN
    rebuild_pruned_cnn,
    read_pruned_cnn_channels,
    build_cnn_keras,
    transfer_cnn_weights,
    # GRU
    rebuild_pruned_gru,
    read_pruned_gru_channels,
    build_gru_keras,
    transfer_gru_weights,
    # Shared
    validate_weight_transfer,
    convert_to_tflite_int8,
    compile_edgetpu,
    evaluate_tflite,
    save_predictions_csv,
    upsert_excel_row,
)

set_seeds(42)
pt_device = torch.device('cpu')  # weight export only

print(f'PyTorch    : {torch.__version__}')
print(f'TensorFlow : {tf.__version__}')

## 2. Configuration

**Edit only this cell.** `MODEL_NAME`, `APPLIANCE_NAME` and `PRUNING_RATIO` must match
the settings used in `03_pruning.ipynb`.

In [ ]:
# ============================================================================
# USER CONFIGURATION  (must match 03_pruning.ipynb settings)
# ============================================================================
MODEL_NAME      = 'cnn'      # 'cnn'  |  'tcn'  |  'gru'
APPLIANCE_NAME  = 'boiler'   # 'boiler'  |  'ac_1'  |  'washing_machine'
PRUNING_RATIO   = 0.75       # target *parameter* reduction — same value used in 03_pruning.ipynb
FINETUNE_EPOCHS = 5          # must match the value used in 03_pruning.ipynb
N_CALIB_BATCHES = 100        # batches from training set used for INT8 calibration
DATASET_NAME    = 'plegma'

# ============================================================================
# AUTO-DERIVED — do not edit below this line
# ============================================================================
_TCN_CFG = {
    'window'          : 600,
    'batch_size'      : 50,
    'depth'           : 9,
    'filters'         : [512, 256, 256, 128, 128, 256, 256, 256, 512],
    'dropout'         : 0.2,
    'stacks'          : 1,
    'args_window_size': 600,
}

_CNN_CFG = {
    'window'          : 299,
    'batch_size'      : 128,
    'args_window_size': 1,
}

_GRU_CFG = {
    'window'          : 199,
    'batch_size'      : 1200,
    'args_window_size': 1,
}

if MODEL_NAME == 'tcn':
    cfg = _TCN_CFG
elif MODEL_NAME == 'gru':
    cfg = _GRU_CFG
else:
    cfg = _CNN_CFG

# Appliance config from src_pytorch (single source of truth — synced with config.py)
app_cfg = get_appliance_params(DATASET_NAME, APPLIANCE_NAME)

WINDOW     = cfg['window']
BATCH_SIZE = cfg['batch_size']
THRESHOLD  = app_cfg['threshold']
CUTOFF     = app_cfg['cutoff']
MIN_ON     = app_cfg.get('min_on')
MIN_OFF    = app_cfg.get('min_off')
MAX_LENGTH = app_cfg.get('max_length')
pct        = int(PRUNING_RATIO * 100)

# Convert user-facing parameter ratio -> internal channel ratio for MetaPruner.
_channel_ratio = param_ratio_to_channel_ratio(PRUNING_RATIO)

# -- Output directories (shared with 03_pruning.ipynb) -----------------------
OUT_DIR     = project_root / 'outputs' / f'{MODEL_NAME}_{APPLIANCE_NAME}'
MODELS_DIR  = OUT_DIR / 'models'       # model checkpoints + .tflite files
PREDS_DIR   = OUT_DIR / 'predictions'  # prediction vs ground-truth CSVs
METRICS_DIR = OUT_DIR / 'metrics'      # metrics CSVs
MODELS_DIR.mkdir(parents=True, exist_ok=True)
PREDS_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

CKPT_PATH  = OUT_DIR / 'checkpoint' / 'model.pt'
DATA_DIR   = project_root / 'data' / 'processed' / DATASET_NAME / APPLIANCE_NAME

# Same comparative-results Excel written by 03_pruning.ipynb
EXCEL_PATH = OUT_DIR / f'{MODEL_NAME}_{APPLIANCE_NAME}_comparative_results.xlsx'

# Fine-tuned checkpoint produced by 03_pruning.ipynb
finetuned_ckpt = MODELS_DIR / f'{MODEL_NAME}_{APPLIANCE_NAME}_pruned_{pct}pct_finetuned.pt'

print(f'Model                : {MODEL_NAME.upper()}')
print(f'Appliance            : {APPLIANCE_NAME}')
print(f'Target param removal : {pct}%  ->  channel ratio: {_channel_ratio:.4f}')
print(f'Finetune epochs      : {FINETUNE_EPOCHS}')
print(f'Calib batches        : {N_CALIB_BATCHES}')
print(f'Window               : {WINDOW}')
print(f'Threshold            : {THRESHOLD} W  |  Cutoff: {CUTOFF} W')
print(f'Min ON               : {MIN_ON} samples  |  Min OFF: {MIN_OFF} samples')
if MAX_LENGTH:
    print(f'Max length           : {MAX_LENGTH} samples')
print(f'Baseline ckpt        : {CKPT_PATH}  ({"found" if CKPT_PATH.exists() else "NOT FOUND"})')
print(f'FT ckpt              : {finetuned_ckpt}  ({"found" if finetuned_ckpt.exists() else "NOT FOUND"})')
print(f'Models dir           : {MODELS_DIR}')
print(f'Predictions          : {PREDS_DIR}')
print(f'Metrics dir          : {METRICS_DIR}')
print(f'Excel                : {EXCEL_PATH}')

## 3. Load Data

In [9]:
data_loader = SimpleNILMDataLoader(
    data_dir=str(DATA_DIR),
    model_name=MODEL_NAME,
    batch_size=BATCH_SIZE,
    input_window_length=WINDOW,
    train=True,
    num_workers=0,
)

print(f'Train batches : {len(data_loader.train)}')
print(f'Val   batches : {len(data_loader.val)}')
print(f'Test  batches : {len(data_loader.test)}')

Train batches : 185341
Val   batches : 10034
Test  batches : 12710


## 4. Rebuild Pruned PyTorch Model

Reproduces the pruned architecture deterministically for both CNN and TCN:
1. Build a fresh baseline model and load the original checkpoint
2. Re-apply the same pruning (same weights → same channels removed)
3. Load the fine-tuned checkpoint

The resulting `pt_model` is used only to **export weights** to TF/Keras.

In [ ]:
print(f'Rebuilding pruned {MODEL_NAME.upper()} model...')

if MODEL_NAME == 'tcn':
    pt_model = rebuild_pruned_tcn(
        cfg                 = cfg,
        baseline_ckpt       = CKPT_PATH,
        pruning_ratio       = PRUNING_RATIO,
        finetuned_ckpt_path = finetuned_ckpt,
        device              = pt_device,
    )
elif MODEL_NAME == 'gru':
    pt_model = rebuild_pruned_gru(
        cfg                 = cfg,
        baseline_ckpt       = CKPT_PATH,
        pruning_ratio       = PRUNING_RATIO,
        finetuned_ckpt_path = finetuned_ckpt,
        device              = pt_device,
    )
else:  # cnn
    pt_model = rebuild_pruned_cnn(
        cfg                 = cfg,
        baseline_ckpt       = CKPT_PATH,
        pruning_ratio       = PRUNING_RATIO,
        finetuned_ckpt_path = finetuned_ckpt,
        device              = pt_device,
    )

dummy_input = torch.randn(1, WINDOW).to(pt_device)
pruned_params, pruned_macs, pruned_mb = get_model_stats(pt_model, dummy_input)

print(f'\nPruned {MODEL_NAME.upper()} (PyTorch float32):')
print(f'  Parameters : {pruned_params:,}')
print(f'  MACs       : {pruned_macs:,}')
print(f'  Size       : {pruned_mb:.3f} MB')

### 4b. Sanity-check: Evaluate Pruned PyTorch Model

Confirm the fine-tuned pruned PyTorch model achieves expected performance **before** converting
to TF/Keras. This isolates any degradation introduced by the weight transfer.

In [ ]:
from src_pytorch.evaluator import evaluate_model

print(f'Evaluating pruned PyTorch {MODEL_NAME.upper()} on test set...')
pt_metrics, pt_gt, pt_pred, pt_gt_status, pt_pred_status = evaluate_model(
    model                = pt_model,
    data_loader          = data_loader,
    model_name           = MODEL_NAME,
    cutoff               = CUTOFF,
    threshold            = THRESHOLD,
    device               = pt_device,
    input_window_length  = WINDOW,
    min_on               = MIN_ON,
    min_off              = MIN_OFF,
    max_length           = MAX_LENGTH,
)

pt_preds_csv = PREDS_DIR / f'{MODEL_NAME}_{APPLIANCE_NAME}_pruned_{pct}pct_pytorch_preds.csv'
save_predictions_csv(pt_gt, pt_pred, pt_preds_csv, gt_status=pt_gt_status, pred_status=pt_pred_status)

print(f'\nPruned PyTorch {MODEL_NAME.upper()} Results:')
for k, v in pt_metrics.items():
    print(f'  {k:<25}: {v:.4f}')

## 5. Build TF/Keras Model

Channel counts are read directly from `pt_model.state_dict()` so the TF model
matches the pruned architecture exactly.

**TCN** — WaveNet-style gated convolutions, Seq2Seq (output = full window):

| PyTorch | TF/Keras |
|---|---|
| `CausalConv1d(k=2, dil=2^i)` | `Conv1D(2, padding='causal', dilation_rate=2^i)` |
| `signal * gate` | `Multiply()` |
| `torch.cat(skips, dim=1)` | `Concatenate(axis=-1)` |

**CNN** — five plain Conv1d + two Dense layers, Seq2Point (output = scalar):

| PyTorch | TF/Keras |
|---|---|
| `Conv1d(k, no padding)` | `Conv1D(k, padding='valid')` |
| `Linear(in, out)` | `Dense(out)` — weights transposed |

TF uses **channel-last** format `(batch, seq_len, channels)` throughout.

In [ ]:
if MODEL_NAME == 'tcn':
    initial_ch, block_filters = read_pruned_channels(
        pt_model, depth=cfg['depth'], stacks=cfg['stacks']
    )
    print(f'Pruned initial_conv channels : {initial_ch}  (baseline: {cfg["filters"][0]})')
    print(f'Pruned block channels        : {block_filters}')
    print(f'Baseline block channels      : {cfg["filters"]}')

    tf_model = build_tcn_keras(
        initial_ch    = initial_ch,
        block_filters = block_filters,
        depth         = cfg['depth'],
        stacks        = cfg['stacks'],
        dropout_rate  = cfg['dropout'],
        seq_len       = WINDOW,
    )

elif MODEL_NAME == 'gru':
    conv1_ch, gru1_hidden, gru2_hidden, fc1_out = read_pruned_gru_channels(pt_model)
    print(f'Pruned conv1 channels : {conv1_ch}  (baseline: 8)')
    print(f'GRU1 hidden size      : {gru1_hidden}  (baseline: 32)')
    print(f'GRU2 hidden size      : {gru2_hidden}  (baseline: 64)')
    print(f'fc1 output units      : {fc1_out}  (baseline: 64)')

    tf_model = build_gru_keras(
        conv1_ch    = conv1_ch,
        gru1_hidden = gru1_hidden,
        gru2_hidden = gru2_hidden,
        fc1_out     = fc1_out,
        seq_len     = WINDOW,
    )

else:  # cnn
    cnn_filters, dense1_units = read_pruned_cnn_channels(pt_model)
    print(f'Pruned CNN channels : {cnn_filters}  (baseline: [30, 30, 40, 50, 50])')
    print(f'Pruned dense1 units : {dense1_units}  (baseline: 1024)')

    tf_model = build_cnn_keras(
        filters      = cnn_filters,
        seq_len      = WINDOW,
        dense1_units = dense1_units,
    )

tf_model.summary(line_length=90)

## 6. Transfer PyTorch Weights → TF/Keras

**Conv1d/Conv1D:** `pt[out, in, k]` → `tf[k, in, out]`  (`permute(2, 1, 0)`)  
**Linear/Dense:** `pt[out, in]` → `tf[in, out]`  (transpose)

In [ ]:
if MODEL_NAME == 'tcn':
    transfer_weights(tf_model, pt_model.state_dict(), cfg['depth'], cfg['stacks'])
elif MODEL_NAME == 'gru':
    transfer_gru_weights(tf_model, pt_model.state_dict())
else:  # cnn
    transfer_cnn_weights(tf_model, pt_model.state_dict())

In [10]:
## --- WEIGHT TRANSFER DIAGNOSTIC ---
sd = pt_model.state_dict()

# Check conv1: PT [out, in, k] → TF [k, in, out]
pt_conv1 = sd['network.0.weight'].permute(2, 1, 0).numpy()
tf_conv1 = tf_model.get_layer('conv1').get_weights()[0]
print(f"conv1  match={np.allclose(pt_conv1, tf_conv1, atol=1e-5)}")

# Check dense1: PT [out, in] → TF [in, out]
pt_d1 = sd['network.13.weight'].numpy().T
tf_d1 = tf_model.get_layer('dense1').get_weights()[0]
print(f"dense1 match={np.allclose(pt_d1, tf_d1, atol=1e-5)}")

# Single sample forward pass (shape-corrected)
sample_pt, _ = next(iter(data_loader.test))
sample_pt = sample_pt[:1]                                           # (1, 299)
sample_tf = sample_pt.numpy()[..., np.newaxis].astype(np.float32)  # (1, 299, 1)

pt_model.eval()
with torch.no_grad():
    pt_out_single = pt_model(sample_pt).item()

tf_out_single = float(tf_model(sample_tf, training=False).numpy().flatten()[0])

diff = abs(pt_out_single - tf_out_single)
status = 'PASS ✓' if diff < 1e-3 else f'FAIL — diff={diff:.6f}'
print(f"\nSingle sample: PT={pt_out_single:.6f}  TF={tf_out_single:.6f}  {status}")

conv1  match=True
dense1 match=True

Single sample: PT=-0.000434  TF=-0.000434  PASS ✓


## 7. Validate â PyTorch vs TF/Keras

Run both models on the same test batch and compare predictions.
A mean absolute difference < 0.001 (normalised) confirms the weight transfer is correct.

In [11]:
print('Validating weight transfer (PyTorch float32 vs TF/Keras float32)...')
val_result = validate_weight_transfer(pt_model, tf_model, data_loader, pt_device)

Validating weight transfer (PyTorch float32 vs TF/Keras float32)...
  Mean |PT − TF|  : 0.00000000  (normalised units)
  Max  |PT − TF|  : 0.00000000  (normalised units)
  Validation      : PASS  (threshold = 0.001)


## 8. Evaluate TF/Keras Float32 Model on Test Set

Run the TF/Keras model (weights already transferred from the fine-tuned pruned PyTorch model)
on the full test split **before** INT8 quantization.

This acts as a **float32 baseline for the TF conversion**, confirming the weight transfer
preserves real-world NILM performance (not just low numerical diff on a single batch).

> No need to re-run the PyTorch evaluation — results from `03_pruning.ipynb` are already in
> the comparative Excel and pipeline CSV.

In [12]:
from tqdm import tqdm

print('Evaluating TF/Keras float32 model on test set...')

all_preds_tf = []
all_gt_tf    = []

for batch_x, batch_y in tqdm(data_loader.test, desc='TF/Keras inference'):
    x_np = batch_x.numpy().astype(np.float32)
    # CNN DataLoader returns (batch, window); TF model always expects (batch, window, 1)
    if x_np.ndim == 2:
        x_np = x_np[..., np.newaxis]
    out = tf_model(x_np, training=False).numpy()
    all_preds_tf.append(out.flatten())
    all_gt_tf.append(batch_y.numpy().flatten())

preds_norm_tf = np.concatenate(all_preds_tf)
gt_norm_tf    = np.concatenate(all_gt_tf)

gt_tf   = gt_norm_tf    * CUTOFF
pred_tf = preds_norm_tf * CUTOFF

pred_tf[pred_tf < THRESHOLD] = 0
pred_tf[pred_tf > CUTOFF]    = CUTOFF

keras_metrics = compute_metrics(
    gt_tf, pred_tf, THRESHOLD,
    min_on=MIN_ON, min_off=MIN_OFF, max_length=MAX_LENGTH,
)

keras_gt_status   = None
keras_pred_status = None
if MIN_ON is not None and MIN_OFF is not None:
    keras_gt_status   = compute_status(gt_tf,   THRESHOLD, MIN_ON, MIN_OFF, MAX_LENGTH)
    keras_pred_status = compute_status(pred_tf, THRESHOLD, MIN_ON, MIN_OFF, MAX_LENGTH)

keras_preds_csv = PREDS_DIR / f'{MODEL_NAME}_{APPLIANCE_NAME}_pruned_{pct}pct_keras_float32_preds.csv'
save_predictions_csv(
    gt_tf, pred_tf, keras_preds_csv,
    gt_status=keras_gt_status, pred_status=keras_pred_status,
)

print(f'\nTF/Keras Float32 Results:')
for k, v in keras_metrics.items():
    print(f'  {k:<25}: {v:.4f}')

Evaluating TF/Keras float32 model on test set...


TF/Keras inference: 100%|██████████| 12710/12710 [09:08<00:00, 23.17it/s]


Predictions saved : cnn_boiler_pruned_75pct_keras_float32_preds.csv

TF/Keras Float32 Results:
  mae                      : 10.1758
  f1                       : 0.8884
  accuracy                 : 0.9957
  precision                : 0.8035
  recall                   : 0.9934
  tp                       : 27780.0000
  tn                       : 1592080.0000
  fp                       : 6793.0000
  fn                       : 185.0000
  total_gt_energy_wh       : 22142.4785
  total_pred_energy_wh     : 24608.2324
  energy_error_percent     : 11.1359
  f1_complex               : 0.8941


## 9. Convert to TFLite Full-Integer INT8

Three-step process:
1. **Representative dataset** — `N_CALIB_BATCHES` training samples for activation range calibration
2. **Convert** — `TFLITE_BUILTINS_INT8` quantises both weights and activations
3. **Save** `.tflite` file

In [13]:
tflite_path = MODELS_DIR / f'{MODEL_NAME}_{APPLIANCE_NAME}_pruned_{pct}pct_int8.tflite'

tflite_path = convert_to_tflite_int8(
    tf_model        = tf_model,
    data_loader     = data_loader,
    window          = WINDOW,
    n_calib_batches = N_CALIB_BATCHES,
    out_path        = tflite_path,
)

tflite_mb = round(tflite_path.stat().st_size / (1024 ** 2), 3)
print(f'Size reduction : {(1 - tflite_mb / pruned_mb) * 100:.1f}%  '
      f'({pruned_mb:.3f} MB â {tflite_mb:.3f} MB)')

Converting to TFLite INT8 (calibrating with 100 batches)...
INFO:tensorflow:Assets written to: C:\Users\SA1E5~1.ATH\AppData\Local\Temp\tmpqve35n16\assets


INFO:tensorflow:Assets written to: C:\Users\SA1E5~1.ATH\AppData\Local\Temp\tmpqve35n16\assets


Saved artifact at 'C:\Users\SA1E5~1.ATH\AppData\Local\Temp\tmpqve35n16'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 299, 1), dtype=tf.float32, name='input')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  2094772505296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2094772506256: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2094772505872: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2094772506640: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2094772505680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2094772504144: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2094772507408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2094772503760: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2094772507024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2094772507984: TensorSpec(shape=(), dtype=tf.resource, name=None)
  20947725072

c:\Users\s.athanasoulias\AppData\Local\Programs\Python\Python313\Lib\site-packages\tensorflow\lite\python\convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
Calibrating: 100%|██████████| 100/100 [00:48<00:00,  2.06batch/s]


Full-integer INT8 conversion succeeded.

TFLite INT8 model saved : c:\Users\s.athanasoulias\OneDrive - Accenture\Desktop\ENERGIZE-Edge-Optimized-NILM-Framework\outputs\cnn_boiler\models\cnn_boiler_pruned_75pct_int8.tflite
File size               : 3.330 MB
Size reduction : 74.8%  (13.222 MB â 3.330 MB)


## 10. Compile for Edge TPU

The `edgetpu_compiler` analyses the `.tflite` graph and reports:
- **Ops mapped to the TPU** — INT8 ops with supported kernels run at ~4 TOPS
- **Ops falling back to CPU** — unsupported ops (e.g. sigmoid, element-wise ×)

Even with CPU fallback ops, the compiled model is valid for evaluation.

> Install: `sudo apt install edgetpu-compiler` (Linux/Colab) or download from  
> [coral.ai/docs/edgetpu/compiler](https://coral.ai/docs/edgetpu/compiler/)

In [ ]:
edgetpu_path = compile_edgetpu(tflite_path, out_dir=MODELS_DIR)

## 11. Evaluate TFLite INT8 Model (CPU Interpreter)

The TFLite CPU interpreter runs the quantized model sample-by-sample:
1. **Quantize input** float32 → int8 using the input tensor's scale / zero-point
2. **Invoke** the interpreter
3. **Dequantize output** int8 → float32
4. Accumulate predictions and compute NILM metrics

In [14]:
print('Evaluating TFLite INT8 model on CPU interpreter...')
tflite_metrics, gt, pred, gt_status, pred_status = evaluate_tflite(
    tflite_path = tflite_path,
    data_loader = data_loader,
    window      = WINDOW,
    cutoff      = CUTOFF,
    threshold   = THRESHOLD,
    min_on      = MIN_ON,
    min_off     = MIN_OFF,
)

preds_csv = PREDS_DIR / f'{MODEL_NAME}_{APPLIANCE_NAME}_pruned_{pct}pct_int8_preds.csv'
save_predictions_csv(gt, pred, preds_csv, gt_status=gt_status, pred_status=pred_status)

print(f'\nTFLite INT8 Results:')
for k, v in tflite_metrics.items():
    print(f'  {k:<25}: {v:.4f}')

Evaluating TFLite INT8 model on CPU interpreter...
  Input  dtype / shape : int8 [  1 299   1]
  Output dtype / shape : int8 [1 1]
  Input  quant params  : scale=0.052507, zero_point=-119
  Output quant params  : scale=0.003777, zero_point=-99


c:\Users\s.athanasoulias\AppData\Local\Programs\Python\Python313\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
TFLite inference: 100%|██████████| 12710/12710 [08:23<00:00, 25.23it/s]


Predictions saved : cnn_boiler_pruned_75pct_int8_preds.csv

TFLite INT8 Results:
  mae                      : 10.2154
  f1                       : 0.8905
  accuracy                 : 0.9958
  precision                : 0.8072
  recall                   : 0.9930
  tp                       : 27770.0000
  tn                       : 1592238.0000
  fp                       : 6635.0000
  fn                       : 195.0000
  total_gt_energy_wh       : 22142.4785
  total_pred_energy_wh     : 24701.6797
  energy_error_percent     : 11.5579
  f1_complex               : 0.8948


## 12. Save Results to Excel

In [15]:
new_row = {
    'Model'           : f'{MODEL_NAME.upper()} Pruned {pct}% + FT + TFLite INT8',
    'Architecture'    : MODEL_NAME.upper(),
    'Appliance'       : APPLIANCE_NAME,
    'Pruning_Ratio_%' : pct,
    'Params'          : pruned_params,
    'MACs'            : pruned_macs,
    'MB'              : tflite_mb,
    'MAE'             : round(tflite_metrics['mae'],                  4),
    'F1'              : round(tflite_metrics['f1'],                   4),
    'F1_Complex'      : round(tflite_metrics['f1_complex'],           4) if 'f1_complex' in tflite_metrics else '',
    'Precision'       : round(tflite_metrics['precision'],            4),
    'Recall'          : round(tflite_metrics['recall'],               4),
    'Accuracy'        : round(tflite_metrics['accuracy'],             4),
    'Energy_Error_%'  : round(tflite_metrics['energy_error_percent'], 2),
}

upsert_excel_row(new_row, EXCEL_PATH)

Upserting row in existing Excel: c:\Users\s.athanasoulias\OneDrive - Accenture\Desktop\ENERGIZE-Edge-Optimized-NILM-Framework\outputs\cnn_boiler\cnn_boiler_comparative_results.xlsx
Excel updated: c:\Users\s.athanasoulias\OneDrive - Accenture\Desktop\ENERGIZE-Edge-Optimized-NILM-Framework\outputs\cnn_boiler\cnn_boiler_comparative_results.xlsx


In [16]:
# Upsert TFLite metrics into the pipeline metrics CSV (appends / overwrites this stage)
pipeline_csv = METRICS_DIR / f'{APPLIANCE_NAME}_{MODEL_NAME}_pipeline_results.csv'

tflite_stage_label = f'Pruned {pct}% + FT + TFLite INT8'
new_row_df = pd.DataFrame([{
    'Stage'          : tflite_stage_label,
    'MAE'            : round(tflite_metrics['mae'],                        4),
    'F1'             : round(tflite_metrics['f1'],                         4),
    'F1_Complex'     : round(tflite_metrics['f1_complex'], 4) if 'f1_complex' in tflite_metrics else '',
    'Precision'      : round(tflite_metrics['precision'],                  4),
    'Recall'         : round(tflite_metrics['recall'],                     4),
    'Accuracy'       : round(tflite_metrics['accuracy'],                   4),
    'Energy_Error_%' : round(tflite_metrics['energy_error_percent'],        2),
}])

if pipeline_csv.exists():
    existing = pd.read_csv(pipeline_csv)
    existing = existing[existing['Stage'] != tflite_stage_label]  # remove stale row
    updated  = pd.concat([existing, new_row_df], ignore_index=True)
else:
    updated = new_row_df

updated.to_csv(pipeline_csv, index=False)
print(f'Pipeline metrics updated: {pipeline_csv}')
updated


Pipeline metrics updated: c:\Users\s.athanasoulias\OneDrive - Accenture\Desktop\ENERGIZE-Edge-Optimized-NILM-Framework\outputs\cnn_boiler\metrics\boiler_cnn_pipeline_results.csv


,Stage,MAE,F1,F1_Complex,Precision,Recall,Accuracy,Energy_Error_%
0,CNN Baseline,7.3070,0.9298,0.9387,0.8791,0.9868,0.9974,1.39
1,CNN Pruned 75%,48.9987,0.0000,0.0000,0.0000,0.0000,0.9829,100.00
2,CNN Pruned 75% + Fine-tuned 1ep,10.3736,0.8936,0.8965,0.8189,0.9834,0.9960,10.26
3,Pruned 75% + FT + TFLite INT8,10.2154,0.8905,0.8948,0.8072,0.9930,0.9958,11.56


## 13. Summary

In [17]:
C, W = 30, 18
SEP  = '=' * (C + W * 2 + 2)
sep  = '-' * (C + W * 2 + 2)
hfmt = f'{{:<{C}}} {{:>{W}}} {{:>{W}}}'
rfmt = f'{{:<{C}}} {{:>{W}}} {{:>{W}}}'

col_ft     = f'Pruned {pct}% + FT (PT)'
col_tflite = 'TFLite INT8'

print(SEP)
print(f'TFLITE SUMMARY  |  {MODEL_NAME.upper()}  |  {APPLIANCE_NAME}  |  pruning={pct}%')
print(SEP)
print(hfmt.format('Metric', col_ft, col_tflite))
print(sep)

# Recover fine-tuned metrics from the shared Excel
ft_label = f'{MODEL_NAME.upper()} Pruned {pct}% + Fine-tuned {FINETUNE_EPOCHS}ep'
ft_mb = ft_mae = ft_f1 = ft_prec = ft_rec = ft_acc = ft_ee = float('nan')

if EXCEL_PATH.exists():
    _df = pd.read_excel(EXCEL_PATH)
    _ft = _df[_df['Model'] == ft_label]
    if not _ft.empty:
        _ft    = _ft.iloc[0]
        ft_mb  = _ft['MB']
        ft_mae = _ft['MAE']
        ft_f1  = _ft['F1']
        ft_prec = _ft['Precision']
        ft_rec  = _ft['Recall']
        ft_acc  = _ft['Accuracy']
        ft_ee   = _ft['Energy_Error_%']


def rrow(label, a, b):
    print(rfmt.format(label, str(a), str(b)))


rrow('MB',           f'{ft_mb:.3f}',  f'{tflite_mb:.3f}')
print(sep)
rrow('MAE (W)',      f'{ft_mae:.4f}', f'{tflite_metrics["mae"]:.4f}')
rrow('F1',           f'{ft_f1:.4f}',  f'{tflite_metrics["f1"]:.4f}')
rrow('Precision',    f'{ft_prec:.4f}',f'{tflite_metrics["precision"]:.4f}')
rrow('Recall',       f'{ft_rec:.4f}', f'{tflite_metrics["recall"]:.4f}')
rrow('Accuracy',     f'{ft_acc:.4f}', f'{tflite_metrics["accuracy"]:.4f}')
rrow('Energy Err %', f'{ft_ee:.2f}',  f'{tflite_metrics["energy_error_percent"]:.2f}')
print(SEP)
print(f'TFLite INT8 : {tflite_path.name}')
if edgetpu_path and edgetpu_path.exists():
    print(f'Edge TPU    : {edgetpu_path.name}')
print(f'Excel       : {EXCEL_PATH}')
print(f'Preds CSV   : {preds_csv.name}')
print(SEP)

TFLITE SUMMARY  |  CNN  |  boiler  |  pruning=75%
Metric                         Pruned 75% + FT (PT)        TFLite INT8
--------------------------------------------------------------------
MB                                            nan              3.330
--------------------------------------------------------------------
MAE (W)                                       nan            10.2154
F1                                            nan             0.8905
Precision                                     nan             0.8072
Recall                                        nan             0.9930
Accuracy                                      nan             0.9958
Energy Err %                                  nan              11.56
TFLite INT8 : cnn_boiler_pruned_75pct_int8.tflite


NameError: name 'edgetpu_path' is not defined